In [1]:
# CELL 1 — Setup
import duckdb
import pandas as pd

SILVER_PATH = r"C:\Users\Lenovo\Downloads\nyc_taxi_project\silver\yellow_trips_silver.parquet"
GOLD_PATH   = r"C:\Users\Lenovo\Downloads\nyc_taxi_project\gold"

con = duckdb.connect()
print("DuckDB ready!")

DuckDB ready!


In [2]:
# CELL 2 — Most profitable hours
result = con.execute(f"""
    SELECT
        pickup_hour,
        COUNT(*)                        AS total_trips,
        ROUND(AVG(fare_amount), 2)      AS avg_fare,
        ROUND(AVG(tip_percentage), 2)   AS avg_tip_pct,
        ROUND(SUM(total_amount), 2)     AS total_revenue
    FROM '{SILVER_PATH}'
    GROUP BY pickup_hour
    ORDER BY total_revenue DESC
    LIMIT 10
""").df()

print("Top 10 Most Profitable Hours:")
display(result)

Top 10 Most Profitable Hours:


,pickup_hour,total_trips,avg_fare,avg_tip_pct,total_revenue
0,17,197141,18.61,21.89,5650331.57
1,16,184600,19.53,21.78,5475079.16
2,18,203543,16.97,22.51,5436256.13
3,15,185402,19.28,19.36,5094431.88
4,14,180942,19.75,19.27,5088723.81
5,19,182624,17.64,22.40,5024594.87
6,13,168712,18.46,19.38,4467756.92
7,20,157322,18.02,22.87,4254481.91
8,21,153326,18.50,21.76,4238995.51
9,12,160181,17.74,19.51,4104580.28


In [3]:
# CELL 3 — Weekday vs Weekend comparison
result = con.execute(f"""
    SELECT
        CASE
            WHEN pickup_day_of_week IN (5, 6) THEN 'Weekend'
            ELSE 'Weekday'
        END                             AS day_type,
        COUNT(*)                        AS total_trips,
        ROUND(AVG(fare_amount), 2)      AS avg_fare,
        ROUND(AVG(trip_distance_miles), 2) AS avg_distance,
        ROUND(AVG(tip_percentage), 2)   AS avg_tip_pct
    FROM '{SILVER_PATH}'
    GROUP BY day_type
""").df()

print("Weekday vs Weekend:")
display(result)

Weekday vs Weekend:


,day_type,total_trips,avg_fare,avg_distance,avg_tip_pct
0,Weekday,2058376,18.51,3.38,21.64
1,Weekend,824705,18.60,3.52,20.00


In [4]:
# CELL 4 — Top 10 most profitable pickup zones
result = con.execute(f"""
    SELECT
        pickup_location_id,
        COUNT(*)                        AS total_trips,
        ROUND(AVG(fare_amount), 2)      AS avg_fare,
        ROUND(AVG(tip_percentage), 2)   AS avg_tip_pct,
        ROUND(SUM(total_amount), 2)     AS total_revenue
    FROM '{SILVER_PATH}'
    GROUP BY pickup_location_id
    ORDER BY total_revenue DESC
    LIMIT 10
""").df()

print("Top 10 Most Profitable Zones:")
display(result)

Top 10 Most Profitable Zones:


,pickup_location_id,total_trips,avg_fare,avg_tip_pct,total_revenue
0,132,151975,60.72,13.12,11722827.14
1,138,86720,41.49,37.05,5552925.56
2,161,129081,15.26,22.94,3043991.00
3,237,141096,12.35,22.30,2780463.76
4,236,130870,13.13,22.25,2694136.25
5,230,94196,17.38,20.40,2464356.57
6,186,104793,15.55,20.31,2441930.71
7,162,100785,14.89,21.60,2323480.22
8,142,94899,13.60,22.05,2017648.17
9,170,83972,14.83,21.27,1917973.45


In [5]:
# CELL 5 — Best tipping hours
result = con.execute(f"""
    SELECT
        pickup_hour,
        ROUND(AVG(tip_percentage), 2)   AS avg_tip_pct,
        COUNT(*)                        AS total_trips
    FROM '{SILVER_PATH}'
    WHERE payment_type_label = 'Credit Card'
    GROUP BY pickup_hour
    ORDER BY avg_tip_pct DESC
    LIMIT 10
""").df()

print("Hours with Best Tips (Credit Card only):")
display(result)


Hours with Best Tips (Credit Card only):


,pickup_hour,avg_tip_pct,total_trips
0,7,45.72,65503
1,20,27.15,132559
2,16,27.00,148893
3,18,26.80,170934
4,19,26.64,153551
5,17,26.43,163268
6,21,25.66,130000
7,22,25.16,117684
8,3,24.98,20299
9,1,24.88,46767


In [6]:
# CELL 6 — Short vs Long trips comparison
result = con.execute(f"""
    SELECT
        CASE
            WHEN trip_distance_miles < 2  THEN 'Short (< 2 miles)'
            WHEN trip_distance_miles < 5  THEN 'Medium (2-5 miles)'
            WHEN trip_distance_miles < 10 THEN 'Long (5-10 miles)'
            ELSE 'Very Long (10+ miles)'
        END                             AS trip_category,
        COUNT(*)                        AS total_trips,
        ROUND(AVG(fare_amount), 2)      AS avg_fare,
        ROUND(AVG(tip_percentage), 2)   AS avg_tip_pct
    FROM '{SILVER_PATH}'
    GROUP BY trip_category
    ORDER BY avg_fare
""").df()

print("Trip Distance Categories:")
display(result)

Trip Distance Categories:


,trip_category,total_trips,avg_fare,avg_tip_pct
0,Short (< 2 miles),1578085,9.62,22.97
1,Medium (2-5 miles),821021,17.98,18.85
2,Long (5-10 miles),247207,33.27,16.78
3,Very Long (10+ miles),236768,64.50,21.86
